# 00 - Environment probe and official-transfer-client test (NO data download)

**Private compute session - not a dataset mirror.** Raw BraTS data are downloaded only from the official TCIA source into ephemeral session storage (`WORK`) and are never written to notebook output, a Kaggle Dataset, GitHub or the website. B1 basis: owner-approved alternative (amendment v1.0-A1); no TCIA authorization is claimed. Enable **GPU** and **Internet** in the notebook settings. Never paste credentials into a cell; use Kaggle Secrets if a private repository needs a token.

In [ ]:
# PARAMETERS - set before running (no secrets here)
REPO_URL = ""   # https URL of this research repository (GitHub)
COMMIT = ""     # exact commit SHA or tag to run (never a moving branch)
WORK = "/tmp/brats"                     # ephemeral session storage (raw data stay here)
EXPORT = "/kaggle/working/export"       # records/reports only - never raw data
assert REPO_URL and COMMIT, "set REPO_URL and COMMIT first"


In [ ]:
import os
os.makedirs(WORK, exist_ok=True)
os.makedirs(EXPORT, exist_ok=True)
!git clone --quiet {REPO_URL} /kaggle/working/repo
%cd /kaggle/working/repo
!git checkout --quiet {COMMIT}
!git rev-parse HEAD
!pip install --quiet -e ".[io,nnunet]"
!brats-uncertainty verify-protocol

## Compute preflight
Measured GPU/CUDA/VRAM/RAM/disk/packages/internet. `NOT_READY` means this environment cannot run the job; the blockers are listed.

In [ ]:
!brats-uncertainty compute-preflight --work-dir {WORK} --json {EXPORT}/probe_general.json
!brats-uncertainty compute-preflight --job JOB-02 --work-dir {WORK} --json {EXPORT}/probe_JOB-02.json

## Official transfer client (IBM Aspera CLI)
Installs IBM's official command-line client into this ephemeral session (README of IBM/aspera-cli: `gem install aspera-cli`, `ascli config transferd install`) and prints its own help for Faspex 5 public links. **Nothing is transferred here.** Record the exact receive syntax and whether a single folder of a package can be selected; notebook 01 uses that confirmed command.

In [ ]:
!apt-get -qq update && apt-get -qq install -y ruby-full > /dev/null
!gem install aspera-cli --no-document
!ascli config transferd install
!ascli --version
!ascli faspex5 -h 2>&1 | head -200

In [ ]:
!brats-uncertainty compute-preflight --job JOB-01 --work-dir {WORK} --json {EXPORT}/probe_JOB-01.json

## Result
Download `/kaggle/working/export/` (probe JSON files) and give them to the owner's checkout. They contain no data and no credentials.